In [4]:
from typing import TypedDict, Annotated
from dotenv import load_dotenv
import os
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [ ]:
from langchain.chat_models import init_chat_model
from langgraph.graph import START, END, StateGraph
from langgraph.graph.message import add_messages

In [ ]:
## test that if everything works fine or not

llm = init_chat_model(
        model='openai/gpt-oss-120b',
        model_provider='groq'
    )

llm.invoke("What is the capital of Bangladesh?").content

'The capital of Bangladesh is **Dhaka**.'

In [12]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

def chatbot(state: State) -> State:
    return {"messages": [llm.invoke(state["messages"])]}

builder = StateGraph(State)
builder.add_node("chatbot_node", chatbot)

builder.add_edge(START, "chatbot_node")
builder.add_edge("chatbot_node", END)

graph = builder.compile()

In [13]:
message = {"role": "user", "content": "Who walked on the moon for the first time? Print only the name"}
# message = {"role": "user", "content": "What is the latest price of MSFT stock?"}
response = graph.invoke({"messages":[message]})

response["messages"]

[HumanMessage(content='Who walked on the moon for the first time? Print only the name', additional_kwargs={}, response_metadata={}, id='a4de2386-0fc7-43b3-a3f9-4f95f7a1c58a'),
 AIMessage(content='Neil Armstrong', additional_kwargs={'reasoning_content': 'The user asks: "Who walked on the moon for the first time? Print only the name". They want just the name. The answer: Neil Armstrong. Must output only the name, no extra text. According to policy, it\'s allowed. Provide just "Neil Armstrong".'}, response_metadata={'token_usage': {'completion_tokens': 67, 'prompt_tokens': 85, 'total_tokens': 152, 'completion_time': 0.14745611, 'completion_tokens_details': {'reasoning_tokens': 56}, 'prompt_time': 0.004350411, 'prompt_tokens_details': None, 'queue_time': 0.054762979, 'total_time': 0.151806521}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_a09bde29de', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019c0e5f-67

In [ ]:
state = None
while True:
    in_message = input("You: ")
    if in_message.lower() in {"quit","exit"}:
        break
    if state is None:
        state: State = {
            "messages": [{"role": "user", "content": in_message}]
        }
    else:
        state["messages"].append({"role": "user", "content": in_message})

    state = graph.invoke(state)
    print("Bot:", state["messages"][-1].content)
    

Bot: Lionel Messi is an Argentine professional football (soccer) player widely regarded as one of the greatest players in the sport’s history.

**Key points about him:**

| Category | Details |
|----------|---------|
| **Full name** | Lionel Andrés Messi Cuccittini |
| **Born** | 24 June 1987 in Rosario, Argentina |
| **Position** | Primarily a forward/attacking midfielder (often plays as a “false‑9” or “playmaker”) |
| **Club career** | • **FC Barcelona** (2004‑2021): 672 league matches, 474 goals; won 10 La Liga titles, 4 Champions Leagues, 7 Copa del Rey, and 6 Ballon d’Or awards.<br>• **Paris Saint‑Germain (PSG)** (2021‑2023): 32 league goals in 58 matches.<br>• **Inter Miami CF** (2023‑present, MLS). |
| **International career** | Argentina national team (2005‑present): 176 caps, 106 goals (as of Oct 2024). Won the 2021 Copa América, 2022 FIFA World Cup, and the 2023 CONMEBOL–UEFA Nations League. |
| **Individual honors** | • 7× Ballon d’Or (2009, 2010, 2011, 2012, 2015, 2019, 202